# 🎬 CineScope: Core Data, Optimization & Model Training Pipeline
This notebook serves as the interactive testing ground, hyperparameter tuning workspace, and offline pre-computation pipeline for the CineScope Hybrid Recommendation Engine. It executes empirical baseline comparisons, searches parameter spaces using Grid Search Cross-Validation, extracts TF-IDF metadata matrices, and serializes production-ready model artifacts.

### 📦 Step 1: Environment Diagnostics & Dependencies

In [8]:
import sys
import os
from pathlib import Path
from collections import defaultdict
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly, SVD, accuracy
from surprise.model_selection import train_test_split, GridSearchCV

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

Python Version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Pandas Version: 3.0.3
NumPy Version: 2.5.0


### 📂 Step 2: Path Normalization & Data Verification

In [9]:
# Universal project anchors
ROOT_DIR = Path(os.getcwd())
DATA_DIR = Path("C:/Users/ZoroDM/cinescope-fresh/data/ml-latest-small")
MODEL_DIR = Path("C:/Users/ZoroDM/cinescope-fresh/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_FILE = DATA_DIR / 'ratings.csv'
MOVIES_FILE = DATA_DIR / 'movies.csv'

if not RATINGS_FILE.exists() or not MOVIES_FILE.exists():
    raise FileNotFoundError("Dataset targets missing from disk. Ensure data path alignment is correct.")

print(f"Root Project Workspace: {ROOT_DIR}")
print(f"Data Directory Integrity Verified: {DATA_DIR.exists()}")

Root Project Workspace: c:\Users\ZoroDM\cinescope-fresh\notebook
Data Directory Integrity Verified: True


### 📉 Step 3: Empirical Predictive Error Baseline Comparisons (`baseline_cmp`)
Before training our deep architectures, we must quantify the absolute performance floor (random choice) and standard statistical tendencies (user/item biases via ALS) to validate the computational necessity of our final machine learning models.

In [10]:
print("📊 Loading MovieLens rating stream for baseline assessment...")
ratings_df = pd.read_csv(RATINGS_FILE)
reader = Reader(rating_scale=(0.5, 5.0))
baseline_data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)

print("✂️ Partitioning data into unified 80% train / 20% test splits...")
trainset_b, testset_b = train_test_split(baseline_data, test_size=0.2, random_state=42)
baseline_results = {}

# 1. Algorithmic Floor: Random Predictor
algo_random = NormalPredictor()
algo_random.fit(trainset_b)
baseline_results['Random Guess (NormalPredictor)'] = accuracy.rmse(algo_random.test(testset_b), verbose=False)

# 2. Control Statistical Baseline: User-Item Deviations
bsl_options = {'method': 'als', 'n_epochs': 5, 'reg_u': 12, 'reg_i': 5}
algo_baseline = BaselineOnly(bsl_options=bsl_options)
algo_baseline.fit(trainset_b)
baseline_results['Statistical Average (BaselineOnly)'] = accuracy.rmse(algo_baseline.test(testset_b), verbose=False)

# 3. Vanilla Matrix Factorization
algo_vanilla_svd = SVD(random_state=42)
algo_vanilla_svd.fit(trainset_b)
baseline_results['Untuned Matrix Factorization (SVD)'] = accuracy.rmse(algo_vanilla_svd.test(testset_b), verbose=False)

"print('\\n' + '='*60)\n",
print("🏆 INITIAL RMSE PARADIGM PERFORMANCE COMPARISON")
print("="*60)
for model, rmse_val in sorted(baseline_results.items(), key=lambda x: x[1], reverse=True):
    print(f"{model:<40} : {rmse_val:.4f}")
print("="*60)

📊 Loading MovieLens rating stream for baseline assessment...
✂️ Partitioning data into unified 80% train / 20% test splits...
Estimating biases using als...
🏆 INITIAL RMSE PARADIGM PERFORMANCE COMPARISON
Random Guess (NormalPredictor)           : 1.4247
Untuned Matrix Factorization (SVD)       : 0.8807
Statistical Average (BaselineOnly)       : 0.8733


### ⚙️ Step 4: Hyperparameter Optimization and Grid Search (`tune_svd`)
Untuned SVD underperforms basic statistical averages on sparse data matrices. Here, we systematically iterate through controlled coordinate variables ($n\_epochs$, $lr\_all$, $reg\_all$) via 3-Fold Cross-Validation to eliminate overfitting trends.

In [11]:
print("⚙️ Executing Grid Search Cross-Validation over defined hyperparameter partitions...")
param_grid = {
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.05, 0.06, 0.1]
}

grid_search = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
grid_search.fit(baseline_data)

print(f"🏆 Target Tuning Complete. Absolute Best Achieved RMSE: {grid_search.best_score['rmse']:.4f}")
print("💡 Mathematically Optimal Structural Coordinates Found:")
optimal_params = grid_search.best_params['rmse']
print(optimal_params)

⚙️ Executing Grid Search Cross-Validation over defined hyperparameter partitions...
🏆 Target Tuning Complete. Absolute Best Achieved RMSE: 0.8621
💡 Mathematically Optimal Structural Coordinates Found:
{'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


### 🧠 Step 5: Build Content-Based Filtering Matrices (TF-IDF)
We read item-level categorical configurations, string-clean the token fields, compile structural text documents, and extract numerical weighting profiles.

In [12]:
print("Parsing movies dataset metadata...")
movies_df = pd.read_csv(MOVIES_FILE)

# Enforce spacing format updates over genre identifiers
movies_df['genres_space'] = movies_df['genres'].str.replace('|', ' ', regex=False)
movies_df['genres_space'] = movies_df['genres_space'].fillna('')

print("Vectorizing genre matrices with TF-IDF...")
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(movies_df['genres_space'])

print(f"Matrix construction finalized. Structural Shape: {genre_matrix.shape}")

Parsing movies dataset metadata...
Vectorizing genre matrices with TF-IDF...
Matrix construction finalized. Structural Shape: (9742, 23)


### 🤖 Step 6: Train Optimized Collaborative Filtering Parameters
Using the optimal hyperparameters verified during Step 4, we instantiate our final production model and execute structural fit actions across the entirety of our interaction volume.

In [13]:
print("Instantiating final SVD engine using optimized parameter maps...")
svd_engine = SVD(
    n_epochs=optimal_params['n_epochs'],
    lr_all=optimal_params['lr_all'],
    reg_all=optimal_params['reg_all'],
    random_state=42
)

print("Factoring explicit feedback matrices over complete training volume...")
full_trainset = baseline_data.build_full_trainset()
svd_engine.fit(full_trainset)

print("SVD parameter optimization complete.")

Instantiating final SVD engine using optimized parameter maps...
Factoring explicit feedback matrices over complete training volume...
SVD parameter optimization complete.


### 📊 Step 7: Run Evaluation Metrics Suite (Precision, Recall, NDCG)
We evaluate how well the system ranks items using Top-10 restriction bands ($k=10$) on an un-trained data block.

In [14]:
eval_model = SVD(
    n_epochs=optimal_params['n_epochs'],
    lr_all=optimal_params['lr_all'],
    reg_all=optimal_params['reg_all'],
    random_state=42
)
eval_model.fit(trainset_b)
test_predictions = eval_model.test(testset_b)

def eval_pipeline(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    ndcgs = []
    
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
        
        if len(user_ratings) >= 2:
            dcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            user_ratings.sort(key=lambda x: x[1], reverse=True)
            idcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            if idcg > 0:
                ndcgs.append(dcg / idcg)

    print(f"• Mean Precision@{k}: {np.mean(list(precisions.values())):.4f}")
    print(f"• Mean Recall@{k}:    {np.mean(list(recalls.values())):.4f}")
    print(f"• Mean NDCG@{k}:      {np.mean(ndcgs):.4f}")

eval_pipeline(test_predictions)

• Mean Precision@10: 0.7507
• Mean Recall@10:    0.5236
• Mean NDCG@10:      0.8047


### 💾 Step 8: Serialize Final Engine Artifacts
We freeze and dump the configured models into serialized binary streams to prepare for production streaming integration within our Streamlit web interface.

In [15]:
print("Exporting structural data binaries to target storage storage locations...")

with open(MODEL_DIR / 'movies.pkl', 'wb') as f:
    pickle.dump(movies_df, f)

with open(MODEL_DIR / 'genre_matrix.pkl', 'wb') as f:
    pickle.dump(genre_matrix, f)

with open(MODEL_DIR / 'svd_model.pkl', 'wb') as f:
    pickle.dump(svd_engine, f)

print("All binary engine artifacts saved successfully. Ready for Streamlit UI deployment framework.")

Exporting structural data binaries to target storage storage locations...
All binary engine artifacts saved successfully. Ready for Streamlit UI deployment framework.
